# Clustering espacial — Leiden, Louvain y GraphST

Lee el h5ad de salida del notebook 01 (QC, filtrado y normalización) y calcula
el clustering con tres métodos: Leiden y Louvain sobre PCA (sin info espacial),
y GraphST, que sí incorpora la posición de los spots.

BayesSpace (cuarto método, espacial) se calcula aparte en R, en el script 02b,
que lee el h5ad generado aquí y añade su propia columna de clusters.

**Input:** `h5ad_outputs/{SAMPLE_ID}_filtered_normalized.h5ad`  
**Output:** `h5ad_outputs/{SAMPLE_ID}_louvain_leiden_graphst.h5ad`


## Librerías y configuración

In [ ]:
import scanpy as sc
import squidpy as sq
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import os

# cambiar SAMPLE_ID para procesar otra muestra
SAMPLE_ID = 'GIN63'
# SAMPLE_ID = 'GIN65'
# SAMPLE_ID = 'GIN67'
# SAMPLE_ID = 'GIN71'

INPUT_FILE = f'/home/imartinezle/h5ad_outputs/{SAMPLE_ID}_filtered_normalized.h5ad'
OUTPUT_FILE = f'/home/imartinezle/h5ad_outputs/{SAMPLE_ID}_louvain_leiden_graphst.h5ad'
FIG_DIR = f'/home/imartinezle/Figuras/figuras_{SAMPLE_ID}/Clustering/'

os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

sc.settings.verbosity = 3
sc.settings.figdir = FIG_DIR
sc.settings.set_figure_params(dpi=100, facecolor='white')

# GraphST es el unico metodo que usa GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

# paleta unificada del pipeline (misma que en el resto de scripts)
PALETTE_PATOLOGO = {
    'tumor': '#D85A30',
    'stroma': '#1D9E75',
    'stroma_linfos': '#534AB7',
    'unannotated': '#B4B2A9',
}


## Cargar datos

In [ ]:
adata = sc.read_h5ad(INPUT_FILE)
adata = adata[adata.obs['Layer_patologo'] != 'unannotated'].copy()

# lista de colores en el mismo orden que las categorias del objeto, para
# usar Layer_patologo como referencia visual en las grids de mas abajo
# (sc.pl.spatial espera una lista de colores, no un ListedColormap)
categorias_patologo = adata.obs['Layer_patologo'].cat.categories.tolist()
colores_patologo = [PALETTE_PATOLOGO[c] for c in categorias_patologo]

print(adata)
print(f'Spots: {adata.n_obs}')
print(f'Genes: {adata.n_vars}')


## PCA, vecinos y UMAP

Se calculan aquí porque el notebook 01 termina en normalización, sin embeddings.
Leiden y Louvain los usan directamente; GraphST calcula su propio embedding.


In [ ]:
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver='arpack', use_highly_variable=True)
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=30)
sc.tl.umap(adata)

sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True, save=f'_pca_variance_{SAMPLE_ID}.png')
sc.pl.umap(adata, color=['total_counts', 'n_genes_by_counts', 'pct_counts_mt'], ncols=3,
           save=f'_umap_qc_{SAMPLE_ID}.png')


## Leiden

Clustering estándar sobre el grafo de vecinos (PCA), sin información espacial.
No tiene una resolución universal: se explora un rango y se decide comparando
cada partición contra la anotación del patólogo (`Layer_patologo`), que sirve
de referencia cualitativa aunque venga de un corte de H&E adyacente y no sea
ground truth exacto. Es una decisión visual, no un criterio estadístico.


In [ ]:
resoluciones = [0.3, 0.5, 0.7, 0.9, 1.1, 1.3]
fig, axes = plt.subplots(2, 4, figsize=(24, 12))

sc.pl.spatial(adata, color='Layer_patologo', palette=colores_patologo, ax=axes[0][0],
              show=False, title='Pathologist annotation (reference)')

for idx, res in enumerate(resoluciones):
    adata_tmp = adata.copy()
    sc.tl.leiden(adata_tmp, resolution=res, key_added='Leiden_tmp')
    ax = axes[(idx + 1) // 4][(idx + 1) % 4]
    sc.pl.spatial(adata_tmp, color='Leiden_tmp', ax=ax, show=False, title=f'resolution={res}')

axes[1][3].axis('off')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}leiden_resolution_exploration_{SAMPLE_ID}.png', dpi=150, bbox_inches='tight')
plt.show()


**Decisión: resolution = 0.6.**
Con resoluciones bajas (0.3-0.5) el estroma linfocitario queda fusionado con
el estroma general, perdiendo una distinción que sí aparece en la anotación
del patólogo. Con resoluciones altas (1.1-1.3) el tumor se fragmenta en varios
clusters pequeños sin una frontera clara entre ellos en el tejido. 0.6 es el
punto intermedio donde las regiones se separan con más nitidez.


In [ ]:
LEIDEN_RESOLUTION = 0.6

sc.tl.leiden(adata, resolution=LEIDEN_RESOLUTION, key_added='Leiden')
print(adata.obs['Leiden'].value_counts())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sc.pl.umap(adata, color='Leiden', ax=axes[0], show=False, title='Leiden clusters (UMAP)')
sc.pl.spatial(adata, color='Leiden', ax=axes[1], show=False, title='Leiden clusters (spatial)')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}leiden_res{LEIDEN_RESOLUTION}_{SAMPLE_ID}.png', dpi=150, bbox_inches='tight')
plt.show()


## Louvain

Louvain parte del mismo grafo de vecinos que Leiden. Se usa directamente la
misma resolución (0.8) en vez de explorar de nuevo: el objetivo aquí no es
optimizar Louvain por separado, sino comparar ambos métodos en igualdad de
condiciones, sobre el mismo grafo y la misma resolución.


In [ ]:
LOUVAIN_RESOLUTION = LEIDEN_RESOLUTION

sc.tl.louvain(adata, resolution=LOUVAIN_RESOLUTION, key_added='Louvain')
print(adata.obs['Louvain'].value_counts())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sc.pl.umap(adata, color='Louvain', ax=axes[0], show=False, title='Louvain clusters (UMAP)')
sc.pl.spatial(adata, color='Louvain', ax=axes[1], show=False, title='Louvain clusters (spatial)')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}louvain_res{LOUVAIN_RESOLUTION}_{SAMPLE_ID}.png', dpi=150, bbox_inches='tight')
plt.show()


## GraphST

Combina expresión génica y vecindad espacial mediante contrastive learning.
Necesita GPU para tiempos de entrenamiento razonables (~2-5 min en GPU, ~20-30 min en CPU).


In [ ]:
from GraphST import GraphST
from GraphST.utils import clustering

model = GraphST.GraphST(adata, device=device)
adata = model.train()


### Elegir el número de clusters

Mismo criterio que con Leiden: se prueba un rango (5 a 10) y se compara cada
partición contra `Layer_patologo`, buscando el valor a partir del cual añadir
más clusters empieza a subdividir zonas ya homogéneas en vez de separar
regiones biológicamente distintas.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(24, 12))

sc.pl.spatial(adata, color='Layer_patologo', palette=colores_patologo, ax=axes[0][0],
              show=False, title='Pathologist annotation (reference)')

for idx, n in enumerate([5, 6, 7, 8, 9, 10]):
    adata_tmp = adata.copy()
    clustering(adata_tmp, n, method='mclust')
    ax = axes[(idx + 1) // 4][(idx + 1) % 4]
    sc.pl.spatial(adata_tmp, color='domain', ax=ax, show=False, title=f'{n} clusters')

axes[1][3].axis('off')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}graphst_n_clusters_{SAMPLE_ID}.png', dpi=150, bbox_inches='tight')
plt.show()


**Decisión: N_CLUSTERS = 7.**
Con 5-6 clusters se mezclan regiones que en la anotación del patólogo aparecen
separadas (tumor y estroma peritumoral quedan en el mismo dominio). A partir de
8-9 aparecen clusters muy pequeños que subdividen zonas que en la partición de
7 ya se ven homogéneas, sin que corresponda a un cambio evidente de arquitectura
tisular. Además, 7 es un número similar al que resultó de Leiden y Louvain con
resolution=0.8, lo que facilita comparar los tres métodos entre sí.


In [ ]:
N_CLUSTERS = 7

clustering(adata, N_CLUSTERS, method='mclust')
adata.obs['GraphST'] = adata.obs['domain']

print(adata.obs['GraphST'].value_counts())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sc.pl.umap(adata, color='GraphST', ax=axes[0], show=False, title='GraphST clusters (UMAP)')
sc.pl.spatial(adata, color='GraphST', ax=axes[1], show=False, title='GraphST clusters (spatial)')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}graphst_{SAMPLE_ID}.png', dpi=150, bbox_inches='tight')
plt.show()


## Comparación de métodos

Se comparan los tres métodos calculados en este notebook lado a lado. BayesSpace
se añadirá a esta comparación en el script 02b, una vez calculado en R.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 7))
sc.pl.spatial(adata, color='Leiden', ax=axes[0], show=False, title='Leiden')
sc.pl.spatial(adata, color='Louvain', ax=axes[1], show=False, title='Louvain')
sc.pl.spatial(adata, color='GraphST', ax=axes[2], show=False, title='GraphST')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}comparacion_metodos_{SAMPLE_ID}.png', dpi=150, bbox_inches='tight')
plt.show()


**Decisión de método de referencia: GraphST.**
Leiden y Louvain, al no usar información espacial, generan dominios más
ruidosos y con fronteras menos limpias sobre el tejido (se ve en la comparación
de arriba). GraphST da regiones más compactas y contiguas, coherentes con la
arquitectura real del tejido, así que se usa como referencia para anotar la
identidad de los clusters (marcadores) y para los análisis posteriores del
pipeline (InferCNV, cell2location).


## Genes marcadores por cluster

Se usa GraphST como referencia para la anotación de identidad de clusters,
por el motivo explicado arriba.


In [ ]:
CLUSTER_METHOD = 'GraphST'

# Convertimos los números de los clústeres a texto ('0', '1', etc.) para que Scanpy no falle
adata.obs['cluster'] = adata.obs[CLUSTER_METHOD].astype(str).astype('category')

# Validamos si algún clúster se quedó con 1 solo spot (por si acaso)
counts = adata.obs['cluster'].value_counts()
valid_clusters = counts[counts >= 2].index
if len(valid_clusters) < len(counts):
    adata = adata[adata.obs['cluster'].isin(valid_clusters)].copy()

sc.tl.rank_genes_groups(adata, groupby='cluster', method='wilcoxon', n_genes=20)
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False, save=f'_markers_{SAMPLE_ID}.png')



In [ ]:
marker_genes = {
    'Tumor':    ['EPCAM', 'KRT8', 'KRT18', 'KRT19', 'MKI67'],
    'Stroma':   ['COL1A1', 'COL3A1', 'FAP', 'ACTA2', 'VIM'],
    'Immune':   ['CD3D', 'CD8A', 'CD4', 'CD68'],
    'Vascular': ['PECAM1', 'VWF', 'CLDN5', 'RAMP2'],
}

sc.pl.dotplot(adata, marker_genes, groupby='cluster', save=f'_dotplot_markers_{SAMPLE_ID}.png')


## Guardar objeto

In [ ]:
adata.write_h5ad(OUTPUT_FILE)

print(f'Objeto guardado: {OUTPUT_FILE}')
print(f'Spots: {adata.n_obs}')

columnas_clustering = [c for c in adata.obs.columns if c in ['Leiden', 'Louvain', 'GraphST', 'cluster']]
print(f'Columnas de clustering: {columnas_clustering}')
